# ✨ LivePortrait Studio - Lái Chuyển Động Cho Ảnh Nhân Vật (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Thời gian chạy (Runtime)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm nút ▶️ ở ô lệnh bên dưới -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ tải trọng số thật và lưu vào Drive, **từ lần thứ 2 trở đi sẽ nạp tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title 🚀 Khởi chạy LivePortrait WebUI 1-Click (Tự Động Caching Google Drive)
import os
import shutil
from google.colab import drive
from IPython.display import clear_output

# 1. Gắn kết Google Drive thông minh
print("🔗 Đang kiểm tra kết nối Google Drive...")
if not os.path.exists('/content/drive/MyDrive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Không thể kết nối Drive tự động ({e}). Sẽ chạy trên bộ nhớ tạm của Colab.")

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/LivePortrait"
drive_weights_dir = f"{drive_cache_dir}/pretrained_weights"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)

%cd /content
if not os.path.exists("/content/LivePortrait"):
    print("⚡ Đang tải mã nguồn LivePortrait...")
    !git clone -b dev https://github.com/camenduru/LivePortrait /content/LivePortrait

%cd /content/LivePortrait

# 2. Kiểm tra trọng số THẬT (loại bỏ file con trỏ Git LFS bị lỗi 'v')
drive_test_file = f"{drive_weights_dir}/liveportrait/base_models/appearance_feature_extractor.pth"
is_valid_weights = os.path.exists(drive_test_file) and os.path.getsize(drive_test_file) > 1000000

if is_valid_weights:
    print("🎉 ĐÃ TÌM THẤY TRỌNG SỐ THẬT TRONG GOOGLE DRIVE! Nạp trực tiếp trong 3 giây...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !cp -r "{drive_weights_dir}" /content/LivePortrait/pretrained_weights
else:
    print("⏳ Đang tải trọng số AI thật (~650MB) qua HuggingFace Hub...")
    !rm -rf /content/LivePortrait/pretrained_weights "{drive_weights_dir}"
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='camenduru/LivePortrait', local_dir='/content/LivePortrait/pretrained_weights', local_dir_use_symlinks=False)
    print("💾 Đang lưu bản sao trọng số thật vào Google Drive để lần sau nạp ngay lập tức...")
    os.makedirs(drive_cache_dir, exist_ok=True)
    !cp -r /content/LivePortrait/pretrained_weights "{drive_cache_dir}/"

# 3. Cài đặt thư viện môi trường
print("📦 Đang chuẩn bị môi trường...")
!pip install -q tyro onnxruntime-gpu onnx gradio colorama ffmpeg-python

# 4. Biên dịch module Cython 3D
%cd /content/LivePortrait/src/utils/dependencies/insightface/thirdparty/face3d/mesh/cython
!python setup.py build_ext --inplace

%cd /content/LivePortrait

# 5. Vá lỗi PyTorch 2.6 (Bắt buộc weights_only=False khi nạp trọng số)
!sed -i "s/torch.load(ckpt_path/torch.load(ckpt_path, weights_only=False/g" /content/LivePortrait/src/utils/helper.py

patch_code = '''import torch
_orig_torch_load = torch.load
torch.load = lambda *args, **kwargs: _orig_torch_load(*args, **dict(kwargs, weights_only=False))
'''
with open('/content/LivePortrait/app.py', 'r', encoding='utf-8') as f:
    app_content = f.read()
if '_orig_torch_load' not in app_content:
    with open('/content/LivePortrait/app.py', 'w', encoding='utf-8') as f:
        f.write(patch_code + app_content)

clear_output()
print("🚀 Đang khởi động LivePortrait WebUI...")
!python app.py --share
